# Stage 2 — Instance Segmentation QA

Interactive QA for the watershed step. Calls into `sickling.stage2_instances` —
no logic is duplicated here.

Use this to:
1. Eyeball the 4-panel QA for any FOV in `unet_predictions/`.
2. Re-run with overridden `InstancesConfig` (e.g. different `min_area`,
   `peak_min_distance`) without touching `configs/base.yaml`.
3. Compare drop-reason distributions across FOVs.

In [ ]:
# NOTE: this sys.path patch is a fallback. The proper fix is to run
# `pip install -e .` once at the repo root, after which `sickling` imports
# from anywhere — including notebooks — without any boilerplate.
import sys
from pathlib import Path

project_root = str(Path.cwd().parent)
if project_root not in sys.path:
    sys.path.append(project_root)


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
import matplotlib.pyplot as plt

from sickling.rbc_classification.py_modules.config import load_config
from sickling.rbc_classification.py_modules.stage2_instances.qa import (
    load_raw_image,
    render_qa_for_h5,
    save_qa_figure,
)

cfg = load_config()
paths = cfg.paths.resolved()
print('unet_predictions:', paths.unet_predictions)
print('raw_images      :', paths.raw_images)
print('figures         :', paths.figures)

## Pick a FOV and render QA

In [ ]:
h5s = sorted(paths.unet_predictions.glob('*.h5'))
print(f'{len(h5s)} prediction(s) on disk:')
for p in h5s:
    print(f'  {p.name}')
h5_path = h5s[0]

In [ ]:
# Look up the matching raw image. The U-Net export prefixes the raw stem with
# 'PRED_' — strip it before searching raw_images/.
out_stem = h5_path.stem
raw_stem = out_stem[len('PRED_'):] if out_stem.startswith('PRED_') else out_stem
raw = load_raw_image(raw_stem, paths.raw_images)
print('raw image found:', raw is not None, ' (shape', None if raw is None else raw.shape, ')')

In [ ]:
fig = render_qa_for_h5(h5_path, cfg.instances, cfg.classes, raw_image=raw)
fig.set_size_inches(13, 13)
plt.show()

## Sweep `min_area` to see how the drop distribution shifts

Useful when the bimodal histogram in panel D suggests a different cutoff.

In [ ]:
from sickling.rbc_classification.py_modules.config import InstancesConfig

for min_area in (550, 800):
    instances_cfg = InstancesConfig(**{**cfg.instances.model_dump(), 'min_area': min_area})
    fig = render_qa_for_h5(h5_path, instances_cfg, cfg.classes, raw_image=raw)
    fig.suptitle(f'min_area = {min_area}', fontsize=14)
    plt.show()

## Save the chosen QA panel

In [ ]:
fig = render_qa_for_h5(h5_path, cfg.instances, cfg.classes, raw_image=raw)
save_qa_figure(fig, paths.figures / f'stage2_qa_{out_stem}.png')
print('saved to', paths.figures / f'stage2_qa_{out_stem}.png')